In [4]:
!pip install tf_keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 11.4 MB/s eta 0:00:00 0:00:01


In [ ]:
from datasets import load_dataset
from transformers import (
    BlipProcessor, 
    BlipForConditionalGeneration, 
    TrainingArguments, 
    Trainer
)
from peft import LoraConfig, get_peft_model
from evaluate import load
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

In [3]:
bleu_metric = load("bleu")
rouge_metric = load("rouge")

In [4]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = processor.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = processor.batch_decode(labels, skip_special_tokens=True)
    
    # BLEU avec différentes longueurs de n-grammes
    bleu_results = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[label] for label in decoded_labels],
        max_order=4  # Pour BLEU-1 à BLEU-4
    )
    
    # ROUGE plus détaillé
    rouge_results = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    
    return {
        "bleu-1": bleu_results["precisions"][0],
        "bleu-4": bleu_results["bleu"],
        "rouge1": rouge_results["rouge1"].mid.fmeasure,
        "rougeL": rouge_results["rougeL"].mid.fmeasure
    }

In [5]:
class ImageCaptioningDataset(Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        caption = item["caption"]
        
        # Process image and text
        inputs = self.processor(
            images=image, 
            text=caption, 
            padding="max_length",
            return_tensors="pt",
            truncation=True
        )
        
        # Remove batch dimension and convert to correct types
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = inputs["input_ids"]  # For BLIP, labels are same as input_ids
        
        return inputs

In [6]:
dataset = load_dataset("yemalin/african-fashion")
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42, shuffle=True)

In [7]:
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [8]:
train_dataset = ImageCaptioningDataset(dataset["train"], processor)
eval_dataset = ImageCaptioningDataset(dataset["test"], processor)

In [9]:
# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["query", "value", "key", "dense"]  # Simplified target modules
)

In [10]:
# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


trainable params: 1,929,216 || all params: 249,343,292 || trainable%: 0.7737


/home/pionners03/projects/afroAnnotation/annot_venv/lib/python3.12/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [11]:
training_args = TrainingArguments(
    output_dir="./blip-afro-fashion",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=0.0001,
    fp16=True,
    logging_steps=500,
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="none",
    label_names=["input_ids"],
    eval_strategy="epoch",
    logging_dir="./logs",
    save_strategy="epoch",                   # Enregistre à chaque époque
    save_total_limit=2,                      # Garde max 2 checkpoints
    save_steps=250,
    load_best_model_at_end=True,             # Charge le meilleur modèle à la fin
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 